In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  HUCRE 1/4 — MOTOR                 
# ══════════════════════════════════════════════════════════════════════════
#  NEREYE BAKMALIYIM?
#     Yeni dosya, yeni sutun duzeni  ->  HUCRE 3 (YAPILANDIRMA)
#     Yeni tablo / yeni varlik turu  ->  HUCRE 2 (SEMA & YAZICILAR)
#     Calistirmak ve dogrulamak      ->  HUCRE 4 (EXECUTE)
#
# ══════════════════════════════════════════════════════════════════════════
import re, json, sqlite3, numbers, traceback
from pathlib import Path
from datetime import datetime, date
import pandas as pd

BASE    = Path.cwd()                # notebook'un bulundugu klasor
DOCS    = BASE / "documents"        # kaynak dosyalarin klasoru
DB_PATH = BASE / "hastane_v2.db"    # uretilecek veritabani
SIFIRLA = True                      # True: her calistirmada DB'yi silip bastan kurar
                                    # False: mevcut DB'nin uzerine ekler/gunceller (UPSERT)


RAPOR = {
    "tarih_hatasi": [],   # cozulemeyen tarih metinleri
    "yetim":        [],   # hasta bir doktora baglanamadi (isim eslesmedi)
    "belirsiz":     [],   # ayni isimde birden fazla doktor -> baglanti kurulmadi
    "gecersiz_tc":  [],   # 11 haneli olmayan TC
    "atlanan":      [],   # dosya bulunamadi / TC'siz kayit
}

# ─────────────────────────────────────────────────────────────────────────
#  1. NORMALIZE EDICILER
#     Kirli veriyi temiz veriye ceviren SAF fonksiyonlar (yan etkisi yok,
#     ayni girdi hep ayni ciktiyi verir). Kirliligin cozuldugu tek katman.
#     Hepsi iki dunyayi birden karsilar: Excel'in gercek tipleri (int64,
#     Timestamp) ve txt'nin duz metni.
# ─────────────────────────────────────────────────────────────────────────
def temiz(v):
    """Herhangi bir degeri kirpilmis metne cevirir. Bos / NaN / 'nan' -> None."""
    if v is None: return None
    if isinstance(v, float) and pd.isna(v): return None
    s = str(v).strip()
    return s if s and s.lower() not in ("nan", "nat", "none") else None

def tc_no(v):
    """TC kimlik -> '74875198582' (metin).
    Excel TC'yi int64 verir, txt metin verir; ikisi de ayni cikti olur.
    Neden METIN: kimlik numarasiyla aritmetik yapilmaz, bastaki sifir korunur.
    11 hane degilse RAPOR'a dusurur ama yine de kabul eder (veri kaybolmasin)."""
    if isinstance(v, numbers.Number) and not isinstance(v, bool):
        if pd.isna(v): return None
        s = str(int(v))                       # 46926593240.0 -> '46926593240'
    else:
        s = temiz(v)
        if not s: return None
        s = re.sub(r"\D", "", s)              # bosluk / nokta / tire temizle
    if not s: return None
    if len(s) != 11: RAPOR["gecersiz_tc"].append(s)
    return s

def iso(v):
    """Tarihi ISO metne cevirir: '07.05.2003' -> '2003-05-07'.
    Neden ISO: SQLite'ta metin olarak siralanir ve BETWEEN dogru calisir.
    'gg.aa.yyyy' formatinda siralama yapilamaz."""
    if v is None: return None
    if isinstance(v, (datetime, date)): return v.strftime("%Y-%m-%d")
    s = temiz(v)
    if not s: return None
    for k in ("%d.%m.%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try: return datetime.strptime(s, k).strftime("%Y-%m-%d")
        except ValueError: pass
    RAPOR["tarih_hatasi"].append(s); return None

def sayi(v):
    """Metin icindeki ilk tam sayiyi ceker: '23 yıl'->23, '%96'->96, '6 gün'->6.
    Excel'den gelen int64/float64 ise dogrudan int'e cevirir.
    Neden gerekli: '23 yıl' metin olarak kalsa ORDER BY yanlis siralar."""
    if isinstance(v, bool): return None
    if isinstance(v, numbers.Number):
        return None if pd.isna(v) else int(v)
    s = temiz(v)
    m = re.search(r"-?\d+", s) if s else None
    return int(m.group()) if m else None

def para(v):
    """Turk formatli tutari sayiya cevirir: '148.500 TL' -> 148500.0
    DIKKAT: TR'de nokta binlik ayraci, virgul ondalik. '1.234,56' -> 1234.56
    Excel zaten sayi veriyorsa dokunmadan gecer (yoksa 148500.0 -> 1485000 olurdu)."""
    if isinstance(v, numbers.Number) and not isinstance(v, bool):
        return None if pd.isna(v) else float(v)
    s = temiz(v)
    if not s: return None
    s = re.sub(r"[^\d,.\-]", "", s)
    s = s.replace(".", "").replace(",", ".") if "," in s else s.replace(".", "")
    try: return float(s)
    except ValueError: return None

SUBE_KOD = re.compile(r"\b(SB\d+)\b")
def sube_kodu(v):
    """Sube kodunu metnin icinden ayiklar.
    txt'de   : 'Anadolu Şifa Hastanesi (SB01)' -> 'SB01'
    excel'de : 'SB01' -> 'SB01'  (zaten temiz, aynen doner)
    Ayni kanonik alan, iki kaynakta farkli bicimde -> tek fonksiyon ikisini de cozer."""
    s = temiz(v)
    m = SUBE_KOD.search(s) if s else None
    return m.group(1) if m else s

# ─────────────────────────────────────────────────────────────────────────
#  2. OKUYUCULAR
#     Iki farkli format, TEK ortak sozlesme: her ikisi de
#     {kaynagin kendi sutun adi: deger} sozluklerinden olusan liste doner.
#     Okuma bicimi farkli, cikti bicimi ayni -> hat asagida ayrisdirmaz.
# ─────────────────────────────────────────────────────────────────────────
AYIRICI = re.compile(r"^[ \t]*[-=_]{3,}[ \t]*$")   # '-----' / '=====' satirlari
def _etiket(s):
    """'GÖREV SÜRESİ   ' -> 'GÖREV SÜRESİ'  (bosluklari tekle, buyut, sondaki nokta at)
    Boylece 'POLİKLİNİK GÜN.:' ile 'POLIKLINIK GUN :' ayni anahtara duser."""
    return re.sub(r"\s+", " ", s.strip()).upper().rstrip(".")

def txt_oku(yol, zorunlu, atla=(), **_):
    """TXT okuyucu: dosyayi '-----' satirlarindan bloklara boler,
    her bloktaki 'ETIKET : deger' satirlarini sozluge cevirir.

    Iki onemli davranis:
      1) Ham satirlar '_satirlar' altinda saklanir -> alt listeler (yapilan
         islemler, alinan hizmetler) bu satirlardan cikarilir.
      2) 'zorunlu' alani olmayan bloklar ELENIR -> dosyanin basindaki baslik
         ve sonundaki 'VERI TABANI SONU' bloklari boylece otomatik atilir."""
    bloklar, aktif = [], []
    for satir in Path(yol).read_text(encoding="utf-8", errors="ignore").splitlines():
        if AYIRICI.match(satir):
            if aktif: bloklar.append(aktif); aktif = []
        else: aktif.append(satir)
    if aktif: bloklar.append(aktif)
    kayitlar = []
    for satirlar in bloklar:
        d = {"_satirlar": satirlar}
        for s in satirlar:
            if ":" not in s: continue
            et, _, dg = s.partition(":")
            if dg.strip(): d[_etiket(et)] = dg.strip()
        v = temiz(d.get(zorunlu))
        if v and v.upper() not in atla:      # baslik / altbilgi / TOPLAM ele
            kayitlar.append(d)
    return kayitlar

def excel_oku(yol, zorunlu, atla=(), **opts):
    """EXCEL okuyucu: her satiri {sutun adi: hucre degeri} sozlugune cevirir.

    KRITIK: hucre degerleri GERCEK TIPIYLE gelir (int64, float, Timestamp) —
    stringe DUZLESTIRILMEZ. Tip kaybi olmadigi icin normalize ediciler dogru
    calisir. (RAG hattindaki parse_excel bunun tersini yapiyor, oradan
    beslenmemenin sebebi bu.)

    'opts' registry'den gelir: kirli dosyalar icin skiprows / sheet_name /
    header / usecols gibi pandas secenekleri."""
    df = pd.read_excel(yol, **opts).dropna(how="all")
    out = []
    for r in df.to_dict("records"):
        v = temiz(r.get(zorunlu))
        if v and v.upper() not in atla: out.append(r)
    return out

# Format -> okuyucu esleme tablosu.
# YENI FORMAT (csv, json...) EKLERSEN: fonksiyonu yaz, buraya 1 satir ekle.
# Boylece hicbir yerde if/elif zinciri olusmaz.
OKUYUCULAR = {"txt": txt_oku, "excel": excel_oku}

# ─────────────────────────────────────────────────────────────────────────
#  3. GENEL SATIR AYRISTIRICILARI
#     Numarali liste satirlarini cozerler. Hangi dosyadan/sutundan geldigini
#     BILMEZLER — o bilgi HUCRE 3'teki sarmalayicilarda (islem_xls vb.).
#     Boylece ayni ayristirici hem txt blogunda hem excel hucresinde calisir.
# ─────────────────────────────────────────────────────────────────────────
ISLEM_TXT  = re.compile(r"^\s*\d+\.\s*(\d{2}\.\d{2}\.\d{4})\s*-\s*(.+?)\s*$")  # "1. 01.01.2026 - Ablasyon"
HIZMET_TXT = re.compile(r"^\s*\d+\.\s*(.+?)\s*$")                              # "1. Koroner Anjiyografi"

def _islem_satirlari(satirlar):
    """Tarihli numarali satirlardan {tarih, islem} uretir (doktor islemleri)."""
    for s in satirlar:
        m = ISLEM_TXT.match(s)
        if m: yield {"tarih": iso(m.group(1)), "islem": m.group(2).strip()}

def _hizmet_satirlari(satirlar):
    """Tarihsiz numarali satirlardan {hizmet} uretir (hastanin aldigi hizmetler).
    Tarihli satirlari atlar, cunku onlar hizmet degil islem kaydidir."""
    for s in satirlar:
        if ISLEM_TXT.match(s): continue          # tarihli satir hizmet degil
        m = HIZMET_TXT.match(s)
        if m: yield {"hizmet": m.group(1).strip()}

# ─────────────────────────────────────────────────────────────────────────
#  4. ESLEME MOTORU
#     Ham kaydi (kaynagin sutun adlariyla) kanonik kayda (bizim sutun
#     adlarimizla) ceviren yer. Mapping'in UC KATMANI burada uygulanir.
# ─────────────────────────────────────────────────────────────────────────
def _ters(spec, kanonik):
    """kanonik sutun adi -> kaynak sutun adi.
    Neden gerekli: donusum fonksiyonlarina (sayi, para, iso) metne cevrilmis
    degil HAM deger gitmeli. Excel'in int64'u bozulmadan ulassin diye."""
    for kay, kan in spec["alanlar"].items():
        if kan == kanonik: return kay
    return kanonik

def uygula(ham, spec):
    """Ham kayit -> kanonik sozluk. Uc katman sirayla:
        1) alanlar : sutun adini degistirir   'Uzmanlık Alanı' -> uzmanlik
        2) donusum : degeri normalize eder    '23 yıl'         -> 23
        3) turetme : 1 alandan N alan uretir  'YILLIK İZİN'    -> 3 sutun
    Bu uc katmanin tanimi HUCRE 3'teki SPEC sozluklerinde."""
    k = {kan: temiz(ham.get(kay)) for kay, kan in spec["alanlar"].items()}
    for alan, fn in spec.get("donusum", {}).items():
        k[alan] = fn(ham.get(_ters(spec, alan), k.get(alan)))
    if spec.get("turetme"): k.update(spec["turetme"](ham))
    return k

# ─────────────────────────────────────────────────────────────────────────
#  5. LOOKUP CACHE
#     FK cozumleri bellekte yapilir: kayit basina SELECT yerine TOPLAM 2 sorgu.
#     kur() her calismada bu cache'leri sifirlar.
# ─────────────────────────────────────────────────────────────────────────
_SUBE  = set()      # {sube_kodu, ...}
_DR_AD = {}         # {ad_soyad: [tc, ...]}

def _sube_var(cur, kod):
    """Sube kodu subeler tablosunda gercekten var mi?
    Yoksa None doner -> FK ihlali yerine NULL yazilir, hat durmaz."""
    if not kod: return None
    if not _SUBE:
        cur.execute("SELECT sube_kodu FROM subeler")
        _SUBE.update(r[0] for r in cur.fetchall())
    return kod if kod in _SUBE else None

def _doktor_tc(cur, ad, hasta_tc):
    """Hastanin doktoruna baglanti kurar.
    SORUN: hasta kaydinda doktorun TC'si YOK, sadece adi var. O yuzden
    baglanti ISIMLE kuruluyor — bu hattaki en kirilgan nokta.
    Birden fazla doktor ayni adi tasiyorsa TAHMIN ETMEZ: RAPOR['belirsiz']'e
    yazar ve NULL birakir. Sessizce yanlis doktora baglamaktan iyidir."""
    if not ad: return None
    if not _DR_AD:
        cur.execute("SELECT ad_soyad, tc FROM doktorlar")
        for a, t in cur.fetchall(): _DR_AD.setdefault(a, []).append(t)
    adaylar = _DR_AD.get(ad, [])
    if len(adaylar) == 1: return adaylar[0]
    if len(adaylar) > 1:
        RAPOR["belirsiz"].append(f"{ad} ({len(adaylar)} eslesme) <- hasta {hasta_tc}")
    else:
        RAPOR["yetim"].append(f"{hasta_tc} -> {ad}")
    return None

# ─────────────────────────────────────────────────────────────────────────
#  6. HAT (PIPELINE)
#     Butun parcalari birlestiren tek dongu:
#       registry satiri -> okuyucu sec -> oku -> uygula -> yazici sec -> yaz
#
#     NEDEN ASAGIDAKI HUCRELERI REFERANS ALABILIYOR?
#     kur() icindeki KAYNAKLAR / SEMA / YAZICILAR isimleri HUCRE 2 ve 3'te
#     tanimli. Python global degiskenleri TANIMLAMA aninda degil CAGRILMA
#     aninda cozer; kur() ancak HUCRE 4'te cagrildigi icin o an hepsi hazirdir.
#
#     TEK TRANSACTION: commit en sonda. Ortada hata olursa rollback edilir ve
#     hata yukari firlatilir (atomik). Yarim kalmis veritabani olusmaz.
#     Hatayi EXECUTE hucresi yakalar ve BASARILI bayragini False yapar.
# ─────────────────────────────────────────────────────────────────────────
def kur():
    global _SUBE, _DR_AD
    _SUBE = set(); _DR_AD = {}                    # cache'leri sifirla
    for _liste in RAPOR.values(): _liste.clear()  # onceki calismanin raporu silinsin
    if SIFIRLA and DB_PATH.exists(): DB_PATH.unlink()
    conn = sqlite3.connect(DB_PATH); conn.execute("PRAGMA foreign_keys=ON")
    cur = conn.cursor(); cur.executescript(SEMA)
    simdi = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    islenen = "(hicbir dosya acilmadi)"           # hata mesajinda gosterilir
    try:
        for kayit in KAYNAKLAR:                   # registry sirasi = yukleme sirasi
            islenen = kayit["dosya"]
            yol = DOCS / kayit["dosya"]
            if not yol.exists():
                RAPOR["atlanan"].append(f"{kayit['dosya']} yok"); continue
            spec = kayit["spec"]
            hamlar = OKUYUCULAR[spec["okuyucu"]](yol, spec["zorunlu"],
                         atla=spec.get("atla", ()), **kayit.get("opts", {}))
            ctx = {"kaynak_dosya": yol.name, "yuklenme": simdi}
            for ham in hamlar:
                cocuklar = list(spec["cocuk"](ham)) if spec.get("cocuk") else []
                YAZICILAR[kayit["hedef"]](cur, uygula(ham, spec), ctx, cocuklar)
            print(f"  {kayit['dosya']:22} -> {kayit['hedef']:<10} {len(hamlar):3} kayit")
    except Exception:
        conn.rollback(); conn.close()             # yarim DB birakma
        print(f"\n  ✖ YUKLEME DURDU ({islenen}) — rollback yapildi, DB yazilmadi.")
        raise                                     # hatayi yutma: EXECUTE yakalayacak
    conn.commit(); return conn

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  HUCRE 2/4 — SEMA & YAZICILAR      🔧 SADECE YENI VARLIK TURU EKLERKEN 
# ══════════════════════════════════════════════════════════════════════════
#  Burasi veritabaninin SEKLINI tanimlar. Yeni bir DOSYA geldiginde degismez
#  (o HUCRE 3). Sadece yeni bir VARLIK TURU eklerken acilir: randevu, fatura,
#  personel gibi.
#
#  ─── TABLOLAR IKI TURLUDUR ───────────────────────────────────────────────
#  VARLIK  (kim / ne var)   : subeler, doktorlar, hastalar
#      Gercek dunyada var olan seyler. Sabit kimlikleri vardir, az degisirler.
#      Icinde olcum/tarih tutmazlar.
#  OLAY    (ne oldu / ne kadar) : yatislar, doktor_islemleri, hizmetler
#      Gerceklesen seyler. Icinde tarih ve sayisal deger olur, varliklara
#      FK ile baglanirlar.
#  Yeni bir tablo eklerken once bunu sor: "bu bir sey mi, yoksa bir olay mi?"
#
#  ─── ANAHTAR POLITIKASI ──────────────────────────────────────────────────
#  Dogal anahtari olan tabloda VEKIL ANAHTAR KULLANILMAZ:
#      doktorlar.tc          <- TC ulusal kimlik, global benzersiz
#      hastalar.tc           <- ayni
#      subeler.sube_kodu     <- SB01, dosyalar arasi kullanilan gercek kod
#  Kimligi olmayan OLAY tablolarinda vekil anahtar kullanilir:
#      yatislar.yatis_id, hizmetler.hizmet_id, doktor_islemleri.islem_id
#  NOT: doktor_no / hasta_no sisteme HIC ALINMADI — onlar dosyadaki sira
#  numarasiydi, kimlik degil.
#
#  ─── YENI TABLO EKLEME TARIFI (3 adim) ───────────────────────────────────
#    1) SEMA'ya CREATE TABLE ekle (FK'leri yazmayi unutma)
#    2) yaz_<tablo>(cur, k, ctx, cocuklar) fonksiyonunu yaz
#    3) YAZICILAR sozlugune "<tablo>": yaz_<tablo> satirini ekle
#    ...sonra HUCRE 3'te spec + registry satirini yaz.
# ══════════════════════════════════════════════════════════════════════════
SEMA = """
-- ══ VARLIK ══ Sube listesi. Ortak boyut: hem doktorlar hem yatislar buna baglanir.
CREATE TABLE IF NOT EXISTS subeler (
  sube_kodu TEXT PRIMARY KEY,                        -- SB01 (gercek is kodu)
  sube_adi TEXT, sehir TEXT, ilce TEXT,
  acilis_yili INTEGER, yatak_kapasitesi INTEGER, doktor_sayisi INTEGER,
  toplam_personel INTEGER, yillik_hasta INTEGER, doluluk_orani INTEGER,
  bashekim TEXT, yillik_ciro REAL, net_kar REAL, kar_marji REAL,
  memnuniyet INTEGER,
  kaynak_dosya TEXT, yuklenme TEXT); -- izlenebilirlik (provenance)

-- ══ VARLIK ══ Doktor. Kimlik TC; ayni TC hangi dosyadan gelirse gelsin AYNI KISI.
CREATE TABLE IF NOT EXISTS doktorlar (
  tc TEXT PRIMARY KEY,                               -- TC = birincil anahtar
  ad_soyad TEXT NOT NULL, uzmanlik TEXT, unvan TEXT, eposta TEXT,
  sube_kodu TEXT,                                    -- hangi subede calisiyor
  goreve_baslama TEXT, gorev_yil INTEGER,
  izin_hakki INTEGER, izin_kullanilan INTEGER, izin_kalan INTEGER,
  haftalik_nobet INTEGER, poliklinik_gunleri TEXT, memnuniyet INTEGER,
  toplam_islem INTEGER,
  kaynak_dosya TEXT, yuklenme TEXT,
  FOREIGN KEY (sube_kodu) REFERENCES subeler(sube_kodu));

-- ══ OLAY ══ Doktorun yaptigi her islem ayri satir (bire-cok).
CREATE TABLE IF NOT EXISTS doktor_islemleri (
  islem_id INTEGER PRIMARY KEY AUTOINCREMENT,        -- olayin kimligi yok -> vekil
  doktor_tc TEXT NOT NULL, tarih TEXT, islem TEXT,
  FOREIGN KEY (doktor_tc) REFERENCES doktorlar(tc));

-- ══ VARLIK ══ Hasta = KISI. Ziyaret bilgisi burada DEGIL, yatislar'da.
--    Dikkat: burada sube_kodu YOK — kisi bir subeye bagli degildir, ZIYARET baglidir.
CREATE TABLE IF NOT EXISTS hastalar (
  tc TEXT PRIMARY KEY,                               -- TC = birincil anahtar
  ad_soyad TEXT NOT NULL, yas INTEGER, cinsiyet TEXT, kan_grubu TEXT,
  kaynak_dosya TEXT, yuklenme TEXT);

-- ══ OLAY ══ Bir ziyaret / yatis. "Ayni hasta birden fazla kez gelebilir"
--    istegini teknik olarak mumkun kilan sey asagidaki UNIQUE satiridir:
--    anahtar sadece hasta_tc olsaydi ikinci yatis birinciyi EZERDI.
CREATE TABLE IF NOT EXISTS yatislar (
  yatis_id INTEGER PRIMARY KEY AUTOINCREMENT,
  hasta_tc TEXT NOT NULL, doktor_tc TEXT, sube_kodu TEXT,
  giris_tarihi TEXT, cikis_tarihi TEXT, yatis_gun INTEGER,
  tani TEXT, servis TEXT, oda_yatak TEXT, sigorta TEXT,
  fatura REAL, odeme_durumu TEXT, taburcu_durumu TEXT,
  hizmetler_json TEXT, hizmet_sayisi INTEGER,        -- dizi (okumak icin) + sayac
  kaynak_dosya TEXT, yuklenme TEXT,
  UNIQUE (hasta_tc, giris_tarihi),                   -- 1 hasta = N ziyaret
  FOREIGN KEY (hasta_tc)  REFERENCES hastalar(tc),
  FOREIGN KEY (doktor_tc) REFERENCES doktorlar(tc),
  FOREIGN KEY (sube_kodu) REFERENCES subeler(sube_kodu));

-- ══ OLAY ══ hizmetler_json dizisinin SORGULANABILIR hali.
--    Dizi okunakli, tablo sorgulanabilir. Ajan COUNT/GROUP BY icin bunu kullanir.
CREATE TABLE IF NOT EXISTS hizmetler (
  hizmet_id INTEGER PRIMARY KEY AUTOINCREMENT,
  yatis_id INTEGER NOT NULL, sira INTEGER, hizmet TEXT,
  FOREIGN KEY (yatis_id) REFERENCES yatislar(yatis_id));

-- Indeksler: FK sutunlari ve isimle arama (yaz_hasta doktoru adla buluyor).
CREATE INDEX IF NOT EXISTS ix_dr_sube    ON doktorlar(sube_kodu);
CREATE INDEX IF NOT EXISTS ix_dr_ad      ON doktorlar(ad_soyad);
CREATE INDEX IF NOT EXISTS ix_yat_hasta  ON yatislar(hasta_tc);
CREATE INDEX IF NOT EXISTS ix_yat_doktor ON yatislar(doktor_tc);
CREATE INDEX IF NOT EXISTS ix_hiz_yatis  ON hizmetler(yatis_id);

-- ══════════════════════════════════════════════════════════════════════
--  VIEW'LER — Text-to-SQL ajaninin ASIL CALISMA YUZEYI.
--  JOIN'i onceden yaparlar; kucuk bir model 6 tablo + 6 FK yerine 2 duz
--  tabloya bakinca cok daha az hata yapar. Yeni sutun eklersen view'i de
--  guncellemeyi unutma.
-- ══════════════════════════════════════════════════════════════════════
CREATE VIEW IF NOT EXISTS v_yatis_detay AS
SELECT y.yatis_id, h.tc AS hasta_tc, h.ad_soyad AS hasta, h.yas,
       h.cinsiyet, h.kan_grubu, y.giris_tarihi, y.cikis_tarihi, y.yatis_gun,
       y.tani, y.servis, y.fatura, y.odeme_durumu, y.taburcu_durumu,
       y.hizmet_sayisi, y.hizmetler_json,
       d.tc AS doktor_tc, d.ad_soyad AS doktor, d.uzmanlik, s.sube_adi, s.sehir
FROM yatislar y
JOIN hastalar h ON h.tc = y.hasta_tc
LEFT JOIN doktorlar d ON d.tc = y.doktor_tc
LEFT JOIN subeler  s ON s.sube_kodu = y.sube_kodu;

CREATE VIEW IF NOT EXISTS v_doktor_ozet AS
SELECT d.tc, d.ad_soyad, d.uzmanlik, d.unvan, d.gorev_yil,
       d.memnuniyet, s.sube_adi, s.sehir,
       (SELECT COUNT(*) FROM doktor_islemleri i WHERE i.doktor_tc = d.tc) AS islem_sayisi,
       (SELECT COUNT(*) FROM yatislar y WHERE y.doktor_tc = d.tc) AS yatis_sayisi,
       (SELECT ROUND(COALESCE(SUM(y.fatura),0),2) FROM yatislar y
          WHERE y.doktor_tc = d.tc) AS toplam_ciro
FROM doktorlar d
LEFT JOIN subeler s ON s.sube_kodu = d.sube_kodu;
"""

# ══════════════════════════════════════════════════════════════════════════
#  YAZICILAR
#  Her yazici ayni imzayi tasir:  yaz_X(cur, k, ctx, cocuklar)
#      cur      : acik veritabani imleci
#      k        : uygula()'dan cikan KANONIK sozluk (bizim sutun adlarimizla)
#      ctx      : hattin ekledigi koken bilgisi (kaynak_dosya, yuklenme)
#      cocuklar : alt tablo satirlari (islemler / hizmetler)
#
#  Her yazicinin uc sorumlulugu var:
#      1) FK'leri coz (kod gercekten var mi?)
#      2) UPSERT et  (ON CONFLICT ... DO UPDATE -> tekrar calistirilabilirlik)
#      3) Cocuk satirlari once SIL sonra yaz (yoksa her calistirmada katlanir)
#
#  <TABLO>_ALAN demetleri ne ise yarar: adlandirilmis parametreli SQL'de
#  sozlukte EKSIK anahtar olamaz. Kaynakta o alan hic gelmemisse bile
#  {alan: None} ile doldurulup sonra gercek degerler uzerine yazilir.
# ══════════════════════════════════════════════════════════════════════════
SUBE_ALAN = ("sube_kodu","sube_adi","sehir","ilce","acilis_yili","yatak_kapasitesi",
             "doktor_sayisi","toplam_personel","yillik_hasta","doluluk_orani",
             "bashekim","yillik_ciro","net_kar","kar_marji","memnuniyet")

def yaz_sube(cur, k, ctx, _c):
    """Sube yazar. Alt tablosu yok (_c kullanilmaz)."""
    if not k.get("sube_kodu"): return
    cur.execute("""INSERT INTO subeler (sube_kodu,sube_adi,sehir,ilce,acilis_yili,
        yatak_kapasitesi,doktor_sayisi,toplam_personel,yillik_hasta,doluluk_orani,
        bashekim,yillik_ciro,net_kar,kar_marji,memnuniyet,kaynak_dosya,yuklenme)
      VALUES (:sube_kodu,:sube_adi,:sehir,:ilce,:acilis_yili,:yatak_kapasitesi,
        :doktor_sayisi,:toplam_personel,:yillik_hasta,:doluluk_orani,:bashekim,
        :yillik_ciro,:net_kar,:kar_marji,:memnuniyet,:kaynak_dosya,:yuklenme)
      ON CONFLICT(sube_kodu) DO UPDATE SET
        sube_adi=excluded.sube_adi, sehir=excluded.sehir, ilce=excluded.ilce,
        yatak_kapasitesi=excluded.yatak_kapasitesi, doktor_sayisi=excluded.doktor_sayisi,
        yillik_ciro=excluded.yillik_ciro, net_kar=excluded.net_kar,
        kar_marji=excluded.kar_marji, memnuniyet=excluded.memnuniyet,
        kaynak_dosya=excluded.kaynak_dosya, yuklenme=excluded.yuklenme""",
      {**{a: None for a in SUBE_ALAN}, **k, **ctx})

DR_ALAN = ("tc","ad_soyad","uzmanlik","unvan","eposta","goreve_baslama","gorev_yil",
           "izin_hakki","izin_kullanilan","izin_kalan","haftalik_nobet",
           "poliklinik_gunleri","memnuniyet","toplam_islem")

def yaz_doktor(cur, k, ctx, cocuklar):
    """Doktoru ve yaptigi islemleri yazar.
    COALESCE kullanimina dikkat: bir kaynak alani bos biraktiysa, daha once
    baska kaynaktan gelen dolu deger SILINMEZ, korunur."""
    if not k.get("tc"):                              # TC yoksa sisteme alma
        RAPOR["atlanan"].append(f"doktor TC'siz: {k.get('ad_soyad')}"); return
    p = {**{a: None for a in DR_ALAN}, **k, **ctx,
         "sube_kodu": _sube_var(cur, k.get("sube_kodu"))}
    cur.execute("""INSERT INTO doktorlar (tc,ad_soyad,uzmanlik,unvan,eposta,sube_kodu,
        goreve_baslama,gorev_yil,izin_hakki,izin_kullanilan,izin_kalan,haftalik_nobet,
        poliklinik_gunleri,memnuniyet,toplam_islem,kaynak_dosya,yuklenme)
      VALUES (:tc,:ad_soyad,:uzmanlik,:unvan,:eposta,:sube_kodu,:goreve_baslama,
        :gorev_yil,:izin_hakki,:izin_kullanilan,:izin_kalan,:haftalik_nobet,
        :poliklinik_gunleri,:memnuniyet,:toplam_islem,:kaynak_dosya,:yuklenme)
      ON CONFLICT(tc) DO UPDATE SET
        ad_soyad=excluded.ad_soyad, uzmanlik=excluded.uzmanlik, unvan=excluded.unvan,
        eposta=COALESCE(excluded.eposta, doktorlar.eposta),
        sube_kodu=COALESCE(excluded.sube_kodu, doktorlar.sube_kodu),
        goreve_baslama=excluded.goreve_baslama, gorev_yil=excluded.gorev_yil,
        izin_hakki=COALESCE(excluded.izin_hakki, doktorlar.izin_hakki),
        izin_kullanilan=COALESCE(excluded.izin_kullanilan, doktorlar.izin_kullanilan),
        izin_kalan=COALESCE(excluded.izin_kalan, doktorlar.izin_kalan),
        haftalik_nobet=excluded.haftalik_nobet, memnuniyet=excluded.memnuniyet,
        toplam_islem=excluded.toplam_islem,
        kaynak_dosya=excluded.kaynak_dosya, yuklenme=excluded.yuklenme""", p)
    _DR_AD.clear()                                   # yeni doktor eklendi -> isim cache'i tazele
    cur.execute("DELETE FROM doktor_islemleri WHERE doktor_tc=?", (k["tc"],))  # once temizle
    for c in cocuklar:
        cur.execute("INSERT INTO doktor_islemleri (doktor_tc,tarih,islem) VALUES (?,?,?)",
                    (k["tc"], c["tarih"], c["islem"]))

YATIS_ALAN = ("giris_tarihi","cikis_tarihi","yatis_gun","tani","servis","oda_yatak",
              "sigorta","fatura","odeme_durumu","taburcu_durumu")

def yaz_hasta(cur, k, ctx, cocuklar):
    """Tek bir kaynak kaydini UC tabloya dagitir:
        hastalar  (kisi)     -> TC ile, tekrar gelirse gunceller
        yatislar  (ziyaret)  -> hasta_tc + giris_tarihi ile, her ziyaret ayri satir
        hizmetler (alt liste)-> yatisa bagli, once silinip yeniden yazilir
    Bu ayrim sayesinde ayni hasta ikinci kez geldiginde kisi tekrarlanmaz,
    sadece yeni bir yatis satiri acilir."""
    if not k.get("tc"):
        RAPOR["atlanan"].append(f"hasta TC'siz: {k.get('ad_soyad')}"); return
    # 1) KISI  — ayni TC ikinci kez gelirse gunceller, yeni satir acmaz
    cur.execute("""INSERT INTO hastalar (tc,ad_soyad,yas,cinsiyet,kan_grubu,
        kaynak_dosya,yuklenme)
      VALUES (:tc,:ad_soyad,:yas,:cinsiyet,:kan_grubu,:kaynak_dosya,:yuklenme)
      ON CONFLICT(tc) DO UPDATE SET
        ad_soyad=excluded.ad_soyad,
        yas=COALESCE(excluded.yas, hastalar.yas),
        cinsiyet=COALESCE(excluded.cinsiyet, hastalar.cinsiyet),
        kan_grubu=COALESCE(excluded.kan_grubu, hastalar.kan_grubu),
        kaynak_dosya=excluded.kaynak_dosya, yuklenme=excluded.yuklenme""",
      {**{a: None for a in ("yas","cinsiyet","kan_grubu")}, **k, **ctx})

    # 2) DOKTOR bagi — hasta kaydinda doktorun TC'si yok, sadece adi var.
    #    Belirsizlik varsa NULL birakir ve RAPOR'a yazar (bkz. _doktor_tc).
    dtc = _doktor_tc(cur, k.get("doktor_adi"), k["tc"])

    # 3) YATIS (ziyaret) + hizmet listesi.
    #    Hizmetler hem JSON dizi (okunakli) hem ayri tablo (sorgulanabilir) olarak yazilir.
    hl = [c["hizmet"] for c in cocuklar]
    p = {**{a: None for a in YATIS_ALAN}, **k, **ctx,
         "hasta_tc": k["tc"], "doktor_tc": dtc,
         "sube_kodu": _sube_var(cur, k.get("sube_kodu")),
         "hizmetler_json": json.dumps(hl, ensure_ascii=False), "hizmet_sayisi": len(hl)}
    cur.execute("""INSERT INTO yatislar (hasta_tc,doktor_tc,sube_kodu,giris_tarihi,
        cikis_tarihi,yatis_gun,tani,servis,oda_yatak,sigorta,fatura,odeme_durumu,
        taburcu_durumu,hizmetler_json,hizmet_sayisi,kaynak_dosya,yuklenme)
      VALUES (:hasta_tc,:doktor_tc,:sube_kodu,:giris_tarihi,:cikis_tarihi,:yatis_gun,
        :tani,:servis,:oda_yatak,:sigorta,:fatura,:odeme_durumu,:taburcu_durumu,
        :hizmetler_json,:hizmet_sayisi,:kaynak_dosya,:yuklenme)
      ON CONFLICT(hasta_tc,giris_tarihi) DO UPDATE SET
        doktor_tc=COALESCE(excluded.doktor_tc, yatislar.doktor_tc),
        sube_kodu=COALESCE(excluded.sube_kodu, yatislar.sube_kodu),
        cikis_tarihi=excluded.cikis_tarihi, yatis_gun=excluded.yatis_gun,
        tani=excluded.tani, servis=excluded.servis, fatura=excluded.fatura,
        odeme_durumu=excluded.odeme_durumu, taburcu_durumu=excluded.taburcu_durumu,
        hizmetler_json=excluded.hizmetler_json, hizmet_sayisi=excluded.hizmet_sayisi,
        kaynak_dosya=excluded.kaynak_dosya, yuklenme=excluded.yuklenme""", p)
    cur.execute("SELECT yatis_id FROM yatislar WHERE hasta_tc=? AND giris_tarihi=?",
                (k["tc"], k.get("giris_tarihi")))
    yid = cur.fetchone()[0]
    cur.execute("DELETE FROM hizmetler WHERE yatis_id=?", (yid,))   # once temizle
    for i, h in enumerate(hl, 1):
        cur.execute("INSERT INTO hizmetler (yatis_id,sira,hizmet) VALUES (?,?,?)", (yid, i, h))

# Hedef tablo -> yazici esleme tablosu. Registry'deki "hedef" alani buraya bakar.
# 🔧 YENI TABLO EKLERSEN: yazici fonksiyonunu yaz ve buraya 1 satir ekle.
YAZICILAR = {"subeler": yaz_sube, "doktorlar": yaz_doktor, "hastalar": yaz_hasta}

In [3]:
# ══════════════════════════════════════════════════════════════════════════
#  HUCRE 3/4 — YAPILANDIRMA          🔧 YENI DOSYA GELINCE SADECE BURAYI AC
# ══════════════════════════════════════════════════════════════════════════
#  Burasi "hangi dosya, nereye, nasil" sorusunun cevabi. Motor (hucre 1) ve
#  sema (hucre 2) burayi okumaz; onlar sadece burada yazdiklarini uygular.
#
#  ╔═══════════════════════════════════════════════════════════════════════╗
#  ║  TARIF 1 — AYNI FORMATTA YENI DOSYA        (en sik durum, 1 satir)    ║
#  ╚═══════════════════════════════════════════════════════════════════════╝
#    En asagidaki KAYNAKLAR listesine bir satir ekle, baska hicbir sey yapma:
#        {"dosya": "doktorlar_v3.xlsx", "hedef": "doktorlar",
#         "spec": SPEC_DOKTOR_XLS}
#    Ayni SPEC'i tekrar kullaniyorsun cunku sutun duzeni ayni.
#
#  ╔═══════════════════════════════════════════════════════════════════════╗
#  ║  TARIF 2 — YENI SUTUN DUZENI               (yeni SPEC + 1 satir)      ║
#  ╚═══════════════════════════════════════════════════════════════════════╝
#    Gelen dosyada sutun adlari farkliysa (or. "Uzmanlik" yerine "Brans"):
#      1) Asagidaki SPEC'lerden en yakinini kopyala, adini degistir
#      2) "alanlar" sozlugunde SOL tarafi yeni dosyanin sutun adlariyla degistir
#         (SAG taraf = kanonik ad, ASLA degismez — sema ona gore kurulu)
#      3) KAYNAKLAR'a yeni SPEC ile bir satir ekle
#
#  ╔═══════════════════════════════════════════════════════════════════════╗
#  ║  TARIF 3 — KIRLI DOSYA                                                ║
#  ╚═══════════════════════════════════════════════════════════════════════╝
#    Baslik 3. satirda        ->  registry satirina  "opts": {"skiprows": 3}
#    Belirli sayfa            ->  "opts": {"sheet_name": "Veri"}
#    Yeni ozet satiri turu    ->  ATLA kumesine ekle ("ARA TOPLAM" gibi)
#
#  ╔═══════════════════════════════════════════════════════════════════════╗
#  ║  TARIF 4 — YENI VARLIK TURU (randevu, fatura...)                      ║
#  ╚═══════════════════════════════════════════════════════════════════════╝
#    Once HUCRE 2 (tablo + yazici), sonra buraya spec + registry satiri.
#
#  ─── BIR SPEC'IN DORT ANAHTARI ───────────────────────────────────────────
#    okuyucu : "txt" | "excel"      -> hangi okuyucu calisacak
#    zorunlu : bu alan bos olan kayit ATILIR (kimlik alani olmali)
#    alanlar : {kaynak sutun adi -> kanonik sutun adi}      [katman 1: ad]
#    donusum : {kanonik alan -> normalize fonksiyonu}       [katman 2: deger]
#    turetme : ham kayit -> ek kanonik alanlar (1 alan -> N alan) [katman 3]
#    cocuk   : ham kayit -> alt tablo satirlari (islem / hizmet listesi)
#    atla    : bu degerleri tasiyan satirlari ele (TOPLAM vb.)
# ══════════════════════════════════════════════════════════════════════════

# Ozet/toplam satirlarini eleyen kume. Okuyucular 'zorunlu' alanin degerini
# buyuk harfe cevirip burada arar; eslesirse satiri hic almaz.
# 🔧 Yeni bir ozet satiri turu gorursen buraya ekle.
ATLA = {"TOPLAM", "GENEL TOPLAM", "ORTALAMA"}

# ─────────────────────────────────────────────────────────────────────────
#  FORMATA OZEL SARMALAYICILAR
#  Motordaki genel ayristiricilari (_islem_satirlari / _hizmet_satirlari)
#  DOGRU YERE baglarlar. Sutun adini bilen tek yer burasi — bu yuzden motorda
#  degil burada duruyorlar.
#
#  Ayni bilginin iki kaynakta FARKLI SEKILDE durmasi mapping'in en ogretici
#  ornegi:
#      txt'de   -> blok icinde alt satirlar     ("1. 01.01.2026 - Ablasyon")
#      excel'de -> tek hucrede satir satir metin (ayni bicim, hucre icinde)
#  Ikisi de ayni {tarih, islem} sozlugunu uretir -> ayni alt tabloya duser.
# ─────────────────────────────────────────────────────────────────────────
def islem_txt(ham): return _islem_satirlari(ham.get("_satirlar", []))
def islem_xls(ham): return _islem_satirlari(str(ham.get("Yapılan İşlemler (2026)", "")).splitlines())

def hizmet_txt(ham): return _hizmet_satirlari(ham.get("_satirlar", []))
def hizmet_xls(ham): return _hizmet_satirlari(str(ham.get("Alınan Hizmetler", "")).splitlines())

# ─────────────────────────────────────────────────────────────────────────
#  TURETME — tek kaynak alanindan BIRDEN COK kanonik alan
#  Basit ad eslemesiyle cozulemeyen durumlar icin. Sadece txt'de gerekiyor,
#  cunku excel ayni bilgiyi zaten ayri sutunlarda veriyor.
# ─────────────────────────────────────────────────────────────────────────
# "30 gün hak / 21 gün kullanıldı / 9 gün kaldı"  ->  3 ayri sayisal sutun
IZIN = re.compile(r"(\d+)\s*gün hak.*?(\d+)\s*gün kullanıldı.*?(\d+)\s*gün kaldı", re.S)
def turet_doktor_txt(ham):
    m = IZIN.search(ham.get("YILLIK İZİN", "") or "")
    return {"izin_hakki": sayi(m.group(1)), "izin_kullanilan": sayi(m.group(2)),
            "izin_kalan": sayi(m.group(3))} if m else {}

def turet_hasta_txt(ham):
    """'58 / Erkek' -> yas=58, cinsiyet='Erkek' (excel'de bu iki ayri sutun)."""
    s = temiz(ham.get("YAŞ / CİNSİYET")) or ""
    p = [x.strip() for x in s.split("/")]
    return {"yas": sayi(p[0]) if p else None,
            "cinsiyet": p[1] if len(p) > 1 else None}

# ══════════════════════════════════════════════════════════════════════════
#  MAPPING SPEC'LERI
#  Okuma bicimi: {"kaynaktaki sutun adi": "bizim kanonik adimiz"}
#  SOL taraf kaynaga gore degisir, SAG taraf SABITTIR (semadaki sutun adi).
#  Yeni format = yeni bir spec; sag taraf hep ayni kalir.
# ══════════════════════════════════════════════════════════════════════════

SPEC_SUBE = {
    "okuyucu": "excel", "zorunlu": "Şube Kodu", "atla": ATLA,
    "alanlar": {
        "Şube Kodu": "sube_kodu", "Şube Adı": "sube_adi", "Şehir": "sehir",
        "İlçe": "ilce", "Açılış Yılı": "acilis_yili",
        "Yatak Kapasitesi": "yatak_kapasitesi", "Doktor Sayısı": "doktor_sayisi",
        "Toplam Personel": "toplam_personel", "Yıllık Hasta Sayısı": "yillik_hasta",
        "Doluluk Oranı (%)": "doluluk_orani", "Başhekim": "bashekim",
        "Yıllık Ciro (TL)": "yillik_ciro", "Net Kâr (TL)": "net_kar",
        "Kâr Marjı (%)": "kar_marji", "Hasta Memnuniyeti (%)": "memnuniyet",
    },
    "donusum": {"acilis_yili": sayi, "yatak_kapasitesi": sayi, "doktor_sayisi": sayi,
                "toplam_personel": sayi, "yillik_hasta": sayi, "doluluk_orani": sayi,
                "yillik_ciro": para, "net_kar": para, "kar_marji": para, "memnuniyet": sayi},
}

# TXT doktor: alanlar 'ETIKET : deger' satirlarindan gelir.
# Etiketler normalize edilmis halleriyle yazilir — nokta ve fazla bosluk atilmis:
#   dosyada "GÖREV SÜRESİ   :"  ->  burada "GÖREV SÜRESİ"
#   dosyada "POLİKLİNİK GÜN.:"  ->  burada "POLİKLİNİK GÜN"
SPEC_DOKTOR_TXT = {
    "okuyucu": "txt", "zorunlu": "TC KİMLİK NO", "atla": ATLA,
    "alanlar": {
        "TC KİMLİK NO": "tc",              # birincil anahtar — bos olan kayit alinmaz
        "AD SOYAD": "ad_soyad", "UZMANLIK ALANI": "uzmanlik",
        "ÜNVAN (TITLE)": "unvan", "ŞUBE": "sube_kodu",
        "GÖREVE BAŞLAMA": "goreve_baslama", "GÖREV SÜRESİ": "gorev_yil",
        "HAFTALIK NÖBET": "haftalik_nobet", "POLİKLİNİK GÜN": "poliklinik_gunleri",
        "HASTA MEMNUN": "memnuniyet", "E-POSTA": "eposta", "TOPLAM İŞLEM": "toplam_islem",
    },
    "donusum": {"tc": tc_no, "sube_kodu": sube_kodu, "goreve_baslama": iso,
                "gorev_yil": sayi, "haftalik_nobet": sayi, "memnuniyet": sayi,
                "toplam_islem": sayi},
    "turetme": turet_doktor_txt,       # 'YILLIK İZİN' tek satiri -> 3 sutun
    "cocuk": islem_txt,                # blok icindeki numarali islem listesi
}

# EXCEL doktor: ayni kanonik alanlar, farkli sutun adlari.
# Izin bilgisi burada zaten 3 ayri sutun -> 'turetme' GEREKMIYOR.
SPEC_DOKTOR_XLS = {
    "okuyucu": "excel", "zorunlu": "TC Kimlik No", "atla": ATLA,
    "alanlar": {
        "TC Kimlik No": "tc",
        "Ad Soyad": "ad_soyad", "Uzmanlık Alanı": "uzmanlik",
        "Ünvan (Title)": "unvan", "Şube Kodu": "sube_kodu",
        "Göreve Başlama": "goreve_baslama", "Görev Süresi (Yıl)": "gorev_yil",
        "Yıllık İzin Hakkı (Gün)": "izin_hakki", "Kullanılan İzin (Gün)": "izin_kullanilan",
        "Kalan İzin (Gün)": "izin_kalan", "Haftalık Nöbet": "haftalik_nobet",
        "Poliklinik Günleri": "poliklinik_gunleri", "Hasta Memnuniyeti (%)": "memnuniyet",
        "E-posta": "eposta", "Toplam İşlem Sayısı": "toplam_islem",
    },
    "donusum": {"tc": tc_no, "sube_kodu": sube_kodu, "goreve_baslama": iso,
                "gorev_yil": sayi, "izin_hakki": sayi, "izin_kullanilan": sayi,
                "izin_kalan": sayi, "haftalik_nobet": sayi, "memnuniyet": sayi,
                "toplam_islem": sayi},
    "cocuk": islem_xls,                # tek hucre icindeki satirli liste
}

# TXT hasta. Iki nokta dikkat cekici:
#   'İLGİLENEN DR' -> doktor_adi : semada BOYLE BIR SUTUN YOK. Yazici bu adi
#        kullanip doktorun TC'sini bulur, sonra doktor_tc olarak yazar.
#   'YAŞ / CİNSİYET' tek alan -> turetme ile iki sutuna ayrilir.
SPEC_HASTA_TXT = {
    "okuyucu": "txt", "zorunlu": "TC KİMLİK NO", "atla": ATLA,
    "alanlar": {
        "TC KİMLİK NO": "tc",
        "AD SOYAD": "ad_soyad", "KAN GRUBU": "kan_grubu", "ŞUBE": "sube_kodu",
        "GİRİŞ TARİHİ": "giris_tarihi", "ÇIKIŞ TARİHİ": "cikis_tarihi",
        "YATIŞ SÜRESİ": "yatis_gun", "TANI": "tani", "SERVİS": "servis",
        "İLGİLENEN DR": "doktor_adi", "ODA / YATAK": "oda_yatak",
        "SİGORTA TÜRÜ": "sigorta", "TOPLAM FATURA": "fatura",
        "ÖDEME DURUMU": "odeme_durumu", "TABURCU DURUMU": "taburcu_durumu",
    },
    "donusum": {"tc": tc_no, "sube_kodu": sube_kodu, "giris_tarihi": iso,
                "cikis_tarihi": iso, "yatis_gun": sayi, "fatura": para},
    "turetme": turet_hasta_txt,        # 'YAŞ / CİNSİYET' -> yas + cinsiyet
    "cocuk": hizmet_txt,               # 'ALINAN HİZMETLER' alt listesi
}

# EXCEL hasta: yas/cinsiyet zaten ayri sutun -> turetme yok.
SPEC_HASTA_XLS = {
    "okuyucu": "excel", "zorunlu": "TC Kimlik No", "atla": ATLA,
    "alanlar": {
        "TC Kimlik No": "tc",
        "Ad Soyad": "ad_soyad", "Yaş": "yas", "Cinsiyet": "cinsiyet",
        "Kan Grubu": "kan_grubu", "Şube Kodu": "sube_kodu",
        "Giriş Tarihi": "giris_tarihi", "Çıkış Tarihi": "cikis_tarihi",
        "Yatış Süresi (Gün)": "yatis_gun", "Tanı": "tani", "Servis": "servis",
        "İlgilenen Doktor": "doktor_adi", "Oda/Yatak": "oda_yatak",
        "Sigorta Türü": "sigorta", "Toplam Fatura (TL)": "fatura",
        "Ödeme Durumu": "odeme_durumu", "Taburcu Durumu": "taburcu_durumu",
    },
    "donusum": {"tc": tc_no, "sube_kodu": sube_kodu, "giris_tarihi": iso,
                "cikis_tarihi": iso, "yas": sayi, "yatis_gun": sayi, "fatura": para},
    "cocuk": hizmet_xls,
}

# ══════════════════════════════════════════════════════════════════════════
#  REGISTRY — HANGI DOSYA NEREYE GIDECEK
#  Yeni dosya eklerken degistirilen TEK yer burasi.
#
#  Bir satirin dort alani:
#    dosya     : documents/ altindaki dosya adi
#    hedef     : hangi tabloya yazilacak (YAZICILAR sozlugundeki anahtar)
#    spec      : yukaridaki esleme tanimlarindan biri
#    opts      : (istege bagli) pandas okuma secenekleri — {"skiprows": 3} gibi
#
#  ⚠ SIRA ONEMLI: FK'lerin cozulebilmesi icin once ebeveyn tablolar yuklenmeli.
#      1) subeler          (doktorlar ve yatislar buna baglanir)
#      2) doktorlar        (yatislar buna baglanir)
#      3) hastalar         (yatis kayitlari burada uretilir)
#    Hastalari doktorlardan once koyarsan kod HATA VERMEZ, sadece butun
#    doktor_tc alanlari NULL kalir. EXECUTE'taki "FK bos" sayaci bunu yakalar.
# ══════════════════════════════════════════════════════════════════════════
KAYNAKLAR = [
    {"dosya": "subeler.xlsx",      "hedef": "subeler",   "spec": SPEC_SUBE},
    {"dosya": "doktorlar_v2.txt",  "hedef": "doktorlar", "spec": SPEC_DOKTOR_TXT},
    {"dosya": "doktorlar_v2.xlsx", "hedef": "doktorlar", "spec": SPEC_DOKTOR_XLS},
    {"dosya": "hastalar_v2.txt",   "hedef": "hastalar",  "spec": SPEC_HASTA_TXT},
    {"dosya": "hastalar_v2.xlsx",  "hedef": "hastalar",  "spec": SPEC_HASTA_XLS},
    # 🔧 YENI DOSYA BURAYA:
    # {"dosya": "doktorlar_v3.xlsx", "hedef": "doktorlar",
    #  "spec": SPEC_DOKTOR_XLS, "opts": {"skiprows": 2}},
]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  HUCRE 4/4 — EXECUTE                          # ══════════════════════════════════════════════════════════════════════════


HATALAR, UYARILAR = [], []          # kontrollerin biriktigi iki liste
BASARILI = False                    # aksi kanitlanana kadar basarisiz sayilir

try:
    conn = kur()
    cur  = conn.cursor()

    # ─── 1. Satir sayilari: bos tablo = o dosya hic islenmemis demektir ───
    print("\n" + "=" * 62)
    for t in ("subeler","doktorlar","doktor_islemleri","hastalar","yatislar","hizmetler"):
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        n = cur.fetchone()[0]
        print(f"  {t:<18}: {n:4}")
        if n == 0: HATALAR.append(f"{t} tablosu bos")

    # ─── 2. Baglanti sagligi: bu uc sayi da 0 olmali ───
    for ad, sql in (("FK bos dr.sube",  "SELECT COUNT(*) FROM doktorlar WHERE sube_kodu IS NULL"),
                    ("FK bos yatis.dr", "SELECT COUNT(*) FROM yatislar WHERE doktor_tc IS NULL")):
        cur.execute(sql); n = cur.fetchone()[0]
        print(f"  {ad:<18}: {n}")
        if n: HATALAR.append(f"{ad} = {n}")
    cur.execute("PRAGMA foreign_key_check")        # semaya aykiri satir var mi
    n = len(cur.fetchall())
    print(f"  {'FK ihlali':<18}: {n}")
    if n: HATALAR.append(f"FK ihlali = {n}")

    # ─── 3. Sessiz hatalar: burasi bossa yukleme temiz demektir ───
    for ad, lst in RAPOR.items():
        if lst:
            UYARILAR.append(f"{ad} ({len(lst)})")
            print(f"  ! {ad}: {sorted(set(map(str,lst)))[:5]}")

    # ─── 4. TC metin olarak mi saklandi? 'text' yazmali ───
    #     'integer' gorursen tc_no() atlanmis demektir; bastaki sifirlar kaybolur.
    print("\n  TC tip kontrolu (metin olmali):")
    cur.execute("SELECT tc, typeof(tc), ad_soyad FROM doktorlar LIMIT 2")
    for r in cur.fetchall(): print("   ", r)
    for t in ("doktorlar", "hastalar"):
        cur.execute(f"SELECT COUNT(*) FROM {t} WHERE typeof(tc) <> 'text'")
        n = cur.fetchone()[0]
        if n: HATALAR.append(f"{t}.tc metin degil ({n} satir)")

    # ─── 5. Hangi kaynaktan kac kayit geldi (koken dagilimi) ───
    #     Bir dosya beklenenden az kayit getirdiyse burada gorunur.
    print("\n  kaynak_dosya dagilimi (koken bilgisi):")
    cur.execute("SELECT kaynak_dosya, COUNT(*) FROM doktorlar GROUP BY kaynak_dosya")
    print("   doktorlar:", cur.fetchall())
    cur.execute("SELECT kaynak_dosya, COUNT(*) FROM hastalar GROUP BY kaynak_dosya")
    print("   hastalar :", cur.fetchall())

    # ─── 6. VIEW calisiyor mu? (ajanin kullanacagi yuzey) ───
    print("\n  En cok cirosu olan 5 doktor:")
    cur.execute("""SELECT tc, ad_soyad, uzmanlik, sube_adi, yatis_sayisi,
                          islem_sayisi, toplam_ciro
                   FROM v_doktor_ozet ORDER BY toplam_ciro DESC LIMIT 5""")
    satirlar = cur.fetchall()
    for r in satirlar: print("   ", r)
    if not satirlar: HATALAR.append("v_doktor_ozet bos donuyor")

    # ─── 7. Hasta -> doktor baglantisi kurulmus mu? ───
    #     doktor_tc sutunu None cikiyorsa isim eslemesi tutmamis demektir.
    print("\n  Hasta -> doktor bagi (TC ile):")
    cur.execute("SELECT hasta_tc, hasta, doktor_tc, doktor FROM v_yatis_detay LIMIT 3")
    satirlar = cur.fetchall()
    for r in satirlar: print("   ", r)
    if not satirlar: HATALAR.append("v_yatis_detay bos donuyor")

    # ─── 8. Hizmet dizisi dogru ayristirilmis mi? ───
    #     hizmet_sayisi 0 cikiyorsa alt liste regex'i tutmamistir.
    print("\n  Hizmet dizisi ornegi:")
    cur.execute("SELECT hasta, hizmet_sayisi, hizmetler_json FROM v_yatis_detay LIMIT 2")
    for r in cur.fetchall(): print("   ", r)
    cur.execute("SELECT COUNT(*) FROM yatislar WHERE hizmet_sayisi = 0")
    n = cur.fetchone()[0]
    if n: UYARILAR.append(f"hizmetsiz yatis ({n})")

    conn.close()

except Exception as e:               # kur() coktu ya da kontrolun kendisi patladi
    HATALAR.append(f"{type(e).__name__}: {e}")
    traceback.print_exc()

# ─── HUKUM — bu hucrenin tek cumlelik cevabi ─────────────────────────────
BASARILI = not HATALAR
print("\n" + "=" * 62)
if BASARILI:
    ek = f"  ({len(UYARILAR)} uyari: {', '.join(UYARILAR)})" if UYARILAR else ""
    print(f"  ✔ BASARILI BITTI — butun kontroller gecti, DB kullanima hazir.{ek}")
    print(f"  DB: {DB_PATH}")
else:
    print(f"  ✖ BASARISIZ BITTI — {len(HATALAR)} hata:")
    for h in HATALAR: print(f"      - {h}")
    if UYARILAR: print(f"  ! uyarilar: {', '.join(UYARILAR)}")
print("=" * 62)
# Sonraki adim: text_to_sql.ipynb -> DB_URI = "file:hastane_v2.db?mode=ro"

  subeler.xlsx           -> subeler     10 kayit
  doktorlar_v2.txt       -> doktorlar   20 kayit
  doktorlar_v2.xlsx      -> doktorlar   20 kayit
  hastalar_v2.txt        -> hastalar    20 kayit
  hastalar_v2.xlsx       -> hastalar    20 kayit

  subeler           :   10
  doktorlar         :   40
  doktor_islemleri  :  495
  hastalar          :   40
  yatislar          :   40
  hizmetler         :  175
  FK bos dr.sube    : 0
  FK bos yatis.dr   : 0
  FK ihlali         : 0

  TC tip kontrolu (metin olmali):
    ('74875198582', 'text', 'Prof. Dr. Sabri Ergene')
    ('93840654526', 'text', 'Doç. Dr. Nurgül Batur')

  kaynak_dosya dagilimi (koken bilgisi):
   doktorlar: [('doktorlar_v2.txt', 20), ('doktorlar_v2.xlsx', 20)]
   hastalar : [('hastalar_v2.txt', 20), ('hastalar_v2.xlsx', 20)]

  En cok cirosu olan 5 doktor:
    ('68853734060', 'Op. Dr. Sedat Güngören', 'Kalp ve Damar Cerrahisi', 'Başkent Yaşam Hastanesi', 1, 15, 486500.0)
    ('60443828296', 'Op. Dr. Emrah Toprak', 'Kalp ve 